In [ ]:
import sys; sys.path.append("../../"); sys.path.append("../../../.."); sys.path.append("../../../gmsh/"); sys.path.append("../../experiments/"); sys.path.append("../../../")
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

import periodic_unit_helper

In [ ]:
h = 4
w = 6 / np.sqrt(3)
wall_w = 0.3
print("width", w)

ipu, m, marker = periodic_unit_helper.get_parallel_tube_periodic(h, w, wall_w, 0.05)

In [ ]:
finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
visualization.plot_2d_mesh(m, pointList = fusedVtx, width = 10, height = 10)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(False)
viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
def cb(i):
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-8
print(opts.factorizer)

cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
benchmark.report()

In [ ]:
cr.success

In [ ]:
name = "anisotropy"
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  
    
variable = wall_w

In [ ]:
render = viewer.offscreenRenderer(1000, 1000)
render.render()
render.save("{}/{}_render_{}_{}.png".format(result_folder, '', name, variable))

In [ ]:
import importlib, periodic_simulation_setup

In [ ]:
importlib.reload(periodic_simulation_setup)

In [ ]:
points = periodic_simulation_setup.visualize_average_deformation_gradient(ipu, 100, plot_max_r=1, plot_min_r=0, show_figure=True, filename = "{}/average_deformation_gradient_{}_{}.svg".format(result_folder, name, variable))

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, useTFT, disableFusedRegionTFT)
if not allowBending:
    fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
# cr = az_optimizer.optimize()

In [ ]:
stiffness_shift = 1e-15
success = False
for i in range(15):
    try:
        stiffness_values, sampled_alphas, stiffness_coefficient = periodic_simulation_setup.visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = stiffness_shift, fixedVars = [], filename = "{}/stiffness_{}_{}.svg".format(result_folder, name, variable), show_figure=True, plot_min_r=0)
        np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
        np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)
        np.save("{}/stiffness_coefficient_{}_{}.npy".format(result_folder, name, variable), stiffness_coefficient)
        print("Solved using stiffness shift: ", stiffness_shift)
        success = True
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
        stiffness_shift *= 10
if (not success):
    print("Failed to solve stiffness!")


In [ ]:
low_pressure_tag = "low_pressure"

In [ ]:
np.save("{}/scale_factors_{}_{}.npy".format(result_folder, name, variable), get_deformation_scale_factors(az_ipu.ipu))
np.save("{}/kappa_{}_{}.npy".format(result_folder, name, variable), az_ipu.getVars()[-2])

np.save("{}/average_deformation_gradient_matrix_{}_{}.npy".format(result_folder, name, variable), get_deformation_matrix(az_ipu.ipu))

np.save("{}/{}_strain_values_{}_{}.npy".format(result_folder, low_pressure_tag, name, variable), utils.getStrains(az_ipu.ipu.sheet)[:, 0])

np.save("{}/{}_dofs_{}_{}.npy".format(result_folder, low_pressure_tag, name, variable), az_ipu.getVars())

In [ ]:
viewer.saveObj("{}/mesh_{}_{}.obj".format(result_folder, name, variable))

In [ ]:
viewer.saveColorizedObj("{}/color_mesh_{}_{}.obj".format(result_folder, name, variable))